# Dimension 1.1 - Spatiotemporal Distribution Mining
Global Moran's I, LISA clusters, Mann-Kendall trends, NMF disease archetypes.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import pymannkendall as mk

from hdi.config import *
from hdi.data.loaders import load_master_panel
from hdi.features.spatial import (
    load_world_geometries, build_spatial_weights,
    compute_morans_i, compute_lisa, merge_spatial_data
)
from hdi.visualization.themes import apply_theme
from hdi.visualization.maps import choropleth_static, lisa_cluster_map
apply_theme()

## 1. Global Spatial Autocorrelation (Moran's I)

In [ ]:
panel = load_master_panel()
world = load_world_geometries()

# Merge latest year data
latest_year = panel['year'].max()
gdf = merge_spatial_data(panel, latest_year, ['daly_rate'] if 'daly_rate' in panel.columns else [])

if 'daly_rate' in gdf.columns and gdf['daly_rate'].notna().sum() > 10:
    w = build_spatial_weights(gdf)
    moran_result = compute_morans_i(gdf, w, 'daly_rate')
    print(f"Global Moran's I: {moran_result['I']:.4f}")
    print(f"p-value: {moran_result['p_value']:.4f}")
    print(f"z-score: {moran_result['z_score']:.4f}")
else:
    print("DALY rate data not available for spatial analysis")

## 2. LISA Cluster Analysis

In [ ]:
if 'daly_rate' in gdf.columns and gdf['daly_rate'].notna().sum() > 10:
    lisa_df = compute_lisa(gdf, w, 'daly_rate')
    print(lisa_df['cluster'].value_counts())
    lisa_cluster_map(gdf, lisa_df, title=f'LISA Clusters: DALY Rate ({latest_year})',
                     save_name='lisa_daly_rate')
    plt.show()

## 3. Mann-Kendall Trend Tests

In [ ]:
# Test monotonic trends per country
if 'daly_rate' in panel.columns:
    trends = []
    for iso3 in panel['iso3'].unique():
        ts = panel[panel['iso3'] == iso3].sort_values('year')['daly_rate'].dropna()
        if len(ts) >= 10:
            result = mk.original_test(ts)
            trends.append({
                'iso3': iso3,
                'trend': result.trend,
                'p_value': result.p,
                'slope': result.slope,
                'tau': result.Tau,
            })
    trends_df = pd.DataFrame(trends)
    print(f"Countries analyzed: {len(trends_df)}")
    print(trends_df['trend'].value_counts())

## 4. NMF Disease Profile Archetypes

In [ ]:
from sklearn.decomposition import NMF

# Construct country x disease matrix (latest year)
# disease_cols = [c for c in panel.columns if c.endswith('_daly_rate')]
# if disease_cols:
#     latest = panel[panel['year'] == latest_year][['iso3'] + disease_cols].dropna()
#     X = latest[disease_cols].values
#     nmf = NMF(n_components=4, random_state=42, max_iter=500)
#     W = nmf.fit_transform(X)  # country loadings
#     H = nmf.components_        # disease loadings
#     print(f"NMF reconstruction error: {nmf.reconstruction_err_:.2f}")
print("NMF analysis: uncomment when disease-specific DALY columns are available")

## 5. Animated Choropleth (DALY Evolution)

In [ ]:
from hdi.visualization.maps import choropleth_plotly

if 'daly_rate' in panel.columns:
    fig = choropleth_plotly(
        panel[panel['daly_rate'].notna()],
        value_col='daly_rate',
        title='DALY Rate Evolution'
    )
    fig.show()